# AguaSegura AI — IA que predice y reduce el agua de enfriamiento de centros de datos

**Qué hace este notebook (de principio a fin):**
1. Baja **clima real** horario (Open-Meteo) de 7 ciudades de México (2022–2026).
2. Simula un centro de datos **híbrido (modo seco / modo evaporativo)**, calibrado con el kit CIFOX.
3. Entrena un **Transformer de atención** y un **LSTM** que pronostican 24 h de agua, energía y temperatura de entrada, con incertidumbre (P10/P50/P90).
4. Usa ese pronóstico para **decidir hora por hora** el modo de enfriamiento y lo prueba en **2026 (clima real que el modelo nunca vio)**.
5. Todo con mapas y gráficas interactivas.

> ⚠️ **DATOS SIMULADOS.** El clima es real; la carga de trabajo, los estados de la torre y los consumos son **simulados** (calibrados con el kit CIFOX, que es 100 % sintético). Ningún número es una medición de un centro de datos real.

In [1]:
import sys, json, subprocess, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from aguasegura.sim import CITIES, load_city, fetch_weather, build_all, INLET_MAX
from aguasegura.nn import (TransformerForecaster, LSTMForecaster, Scaler, predict, TARGETS, LH, FH)
from aguasegura.policy import prepare, evaluate, decisions, pareto, POLICIES
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
RETRAIN = True          # True = reentrena (≈2 min en GPU). False = usa models/*.pt ya guardados
print('dispositivo:', dev, '| torch', torch.__version__)
pd.options.display.float_format = '{:,.1f}'.format

dispositivo: cuda | torch 2.11.0+cu128


## 1 · Clima real: dónde es fácil (y difícil) enfriar sin gastar agua

In [2]:
build_all()   # descarga el clima la primera vez (Open-Meteo) y genera la simulación
rows = []
for c, (lat, lon) in CITIES.items():
    d = load_city(c); t = d[d.time >= '2026-01-01']
    rows.append(dict(ciudad=c, lat=lat, lon=lon, T_media=t['T'].mean(), T_max=t['T'].max(), HR_media=t.RH.mean(),
                     bulbo_humedo_p95=t.Twb.quantile(.95), pct_seco_factible=100*(t.inlet_d <= INLET_MAX).mean(),
                     pct_evap_factible=100*(t.inlet_e <= INLET_MAX).mean()))
clima = pd.DataFrame(rows)
fig = px.scatter_map(clima, lat='lat', lon='lon', size='T_media', color='pct_seco_factible', text='ciudad',
        color_continuous_scale='RdYlGn', range_color=(40, 100), size_max=28, zoom=4.3, center=dict(lat=24, lon=-102),
        hover_data={'T_media':':.1f','T_max':':.1f','HR_media':':.0f','pct_seco_factible':':.0f','pct_evap_factible':':.0f','lat':False,'lon':False},
        map_style='carto-darkmatter', title='Clima real 2026 · color = % de horas en que el modo SECO mantiene la entrada ≤ 27 °C')
fig.update_layout(height=560, margin=dict(l=0,r=0,t=50,b=0)); fig.show()
clima.drop(columns=['lat','lon']).round(1)

,ciudad,T_media,T_max,HR_media,bulbo_humedo_p95,pct_seco_factible,pct_evap_factible
0,Querétaro,19.2,34.3,59.3,17.4,82.5,99.9
1,Monterrey,23.9,40.1,63.5,24.0,59.9,39.0
2,Ciudad de México,17.7,32.7,60.1,15.6,93.6,100.0
3,Guadalajara,22.0,35.9,55.7,19.2,75.7,94.4
4,Mérida,26.8,39.6,72.6,26.1,45.7,14.0
5,Chihuahua,22.3,37.9,37.4,19.5,64.2,92.4
6,Mexicali,28.9,48.8,28.7,25.4,39.1,68.5


## 2 · Simulador físico (calibrado con el kit)
Dos modos por hora (se calculan **ambos** para poder evaluar decisiones contrafactuales):
- **Evaporativo:** poca energía, mucha agua. El agua depende de la carga de calor, de la temperatura de la cuenca de la torre (inercia de ~3 h, **estado oculto**) y de los ciclos de concentración (incrustación, **estado oculto** que se limpia cada ~2 semanas).
- **Seco:** casi sin agua, más energía; solo es seguro si la temperatura de entrada se mantiene ≤ 27 °C.

Los estados ocultos son la razón de usar un modelo con memoria: hay que inferirlos de la historia.

In [3]:
kit = pd.read_csv('data/operaciones_simuladas.csv'); kit = kit[kit.scenario == 'base']
def ratios(df, w, e_it, e_tot): return df[w].sum()/df[e_it].sum(), df[e_tot].sum()/df[e_it].sum()
q = load_city('Querétaro')
cal = pd.DataFrame({
  'WUE evap (L/kWh IT)': [ratios(kit[kit.site_id=='SIM-B'],'water_l','energy_it_kwh','energy_total_kwh')[0], q.water_e.sum()/q.IT.sum()],
  'PUE evap': [ratios(kit[kit.site_id=='SIM-B'],'water_l','energy_it_kwh','energy_total_kwh')[1], q.energy_e.sum()/q.IT.sum()],
  'WUE seco (L/kWh IT)': [ratios(kit[kit.site_id=='SIM-A'],'water_l','energy_it_kwh','energy_total_kwh')[0], q.water_d.sum()/q.IT.sum()],
  'PUE seco': [ratios(kit[kit.site_id=='SIM-A'],'water_l','energy_it_kwh','energy_total_kwh')[1], q.energy_d.sum()/q.IT.sum()]},
  index=['kit CIFOX (sintético)', 'simulador + clima real Querétaro']).T
display(cal.round(3))
w = q[(q.time >= '2026-04-20') & (q.time < '2026-04-27')]
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=.06,
        subplot_titles=('Agua por hora: el modo seco casi no gasta', 'Temperatura de entrada del modo seco vs. límite', 'Carga de IA (%) y temperatura exterior'))
fig.add_scatter(x=w.time, y=w.water_e, name='agua evaporativo (L/h)', line_color='#2ca3dc', row=1, col=1)
fig.add_scatter(x=w.time, y=w.water_d, name='agua seco (L/h)', line_color='#7fd18b', row=1, col=1)
fig.add_scatter(x=w.time, y=w.inlet_d, name='entrada modo seco (°C)', line_color='#ff8a5c', row=2, col=1)
fig.add_hline(y=INLET_MAX, line_dash='dash', line_color='red', annotation_text='27 °C', row=2, col=1)
fig.add_scatter(x=w.time, y=w.load, name='carga (%)', line_color='#c9a0ff', row=3, col=1)
fig.add_scatter(x=w.time, y=w['T'], name='T exterior (°C)', line_color='#ffd166', row=3, col=1)
fig.update_layout(height=640, template='plotly_dark', title='Una semana real de abril en Querétaro (modelo: simulador)'); fig.show()

,kit CIFOX (sintético),simulador + clima real Querétaro
WUE evap (L/kWh IT),0.9,1.0
PUE evap,1.2,1.2
WUE seco (L/kWh IT),0.0,0.0
PUE seco,1.4,1.3


## 3 · Los modelos
**Transformer encoder–decoder**: el encoder atiende a las 48 h de historia; el decoder usa las 24 h futuras conocidas (pronóstico del clima con error y carga planificada) y hace *cross-attention* a la historia. Salida: cuantiles P10/P50/P90 de 5 variables × 24 h.
**LSTM**: mismo contrato de entrada/salida, como alternativa clásica.

Entrenamiento: 2022–2024. Validación: 2025. **Prueba: 2026 (clima real nunca visto).** El pronóstico futuro del clima se alimenta **con error** (AR(1), σ_T≈1.2 °C, σ_HR≈6 %) para no hacer trampa.

In [4]:
if RETRAIN:
    p = subprocess.Popen([sys.executable, '-u', 'train.py', '14'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)
    for line in p.stdout:
        if line.startswith(('transformer ep', 'lstm ep', 'ventanas')): print(line.rstrip())
    p.wait()
sc = Scaler.from_json(json.load(open('models/scaler.json')))
tf = TransformerForecaster().to(dev); tf.load_state_dict(torch.load('models/transformer.pt')); tf.eval()
ls = LSTMForecaster().to(dev);       ls.load_state_dict(torch.load('models/lstm.pt')); ls.eval()
n = lambda m: sum(p.numel() for p in m.parameters())
print(f'\nparámetros — Transformer: {n(tf):,} | LSTM: {n(ls):,}')

ventanas train=91819 val=10199 dispositivo=cuda


transformer ep01 train=0.1514 val=0.0634 (4s)


transformer ep02 train=0.0617 val=0.0520 (7s)


transformer ep03 train=0.0536 val=0.0499 (11s)


transformer ep04 train=0.0506 val=0.0484 (14s)


...

lstm ep11 train=0.0479 val=0.0478 (10s)


lstm ep12 train=0.0478 val=0.0477 (11s)


lstm ep13 train=0.0477 val=0.0476 (12s)


lstm ep14 train=0.0476 val=0.0476 (13s)



parámetros — Transformer: 307,279 | LSTM: 234,927


### Resultados en 2026 (clima real que el modelo nunca vio)

In [5]:
M = json.load(open('models/metrics.json')); mae = pd.DataFrame(M['mae']).T
etiq = {'water_e':'Agua evaporativo (L/h)','water_d':'Agua seco (L/h)','energy_e':'Energía evap (kWh/h)','energy_d':'Energía seco (kWh/h)','inlet_d':'Entrada seco (°C)'}
long = mae[list(etiq)].rename(columns=etiq).reset_index().melt('index', var_name='variable', value_name='MAE').rename(columns={'index':'modelo'})
fig = px.bar(long, x='variable', y='MAE', color='modelo', barmode='group', template='plotly_dark',
             color_discrete_sequence=['#8f9aa8', '#ffb347', '#2ca3dc'], title='Error absoluto medio del pronóstico a 24 h (menor es mejor)')
fig.update_layout(height=420); fig.show()
mejora = (1 - mae.loc['Transformer (atención)', list(etiq)] / mae.loc['Persistencia (ayer)', list(etiq)]) * 100
print('Mejora del Transformer sobre persistencia (%):'); print(mejora.rename(etiq).round(0).to_string())
print('\nCobertura real del intervalo P10–P90 (ideal 80 %):', (mae['cobertura_P10_P90'].dropna()*100).round(1).to_dict())
print('Nota: la persistencia usa los valores contrafactuales de ayer (información que un operador real no tiene): ventaja para el baseline.')

Mejora del Transformer sobre persistencia (%):
Agua evaporativo (L/h)   66.0
Agua seco (L/h)          57.0
Energía evap (kWh/h)     78.0
Energía seco (kWh/h)     77.0
Entrada seco (°C)        52.0

Cobertura real del intervalo P10–P90 (ideal 80 %): {'Transformer (atención)': 77.9, 'LSTM': 80.0}
Nota: la persistencia usa los valores contrafactuales de ayer (información que un operador real no tiene): ventaja para el baseline.


## 4 · Explorador interactivo: pronóstico vs. realidad, día por día
Desliza el control para recorrer días de 2026. Arriba: agua evaporativa real vs. pronóstico (banda P10–P90). Centro: temperatura de entrada del modo seco real vs. pronóstico y el límite de 27 °C. Abajo: **decisión de la IA** (seco = 1).

In [6]:
city = 'Querétaro'
P = prepare(city, tf, sc, dev)
dry = decisions(P)[POLICIES[3]]
days = np.linspace(5, len(P['y'])-6, 40).astype(int)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=.07, row_heights=[.4,.4,.2],
        subplot_titles=('Agua modo evaporativo (L/h)', 'Temperatura de entrada modo seco (°C)', 'Decisión de la IA (1 = seco)'))
for k, j in enumerate(days):
    t = pd.to_datetime(P['times'][j]); vis = k == 0
    p = P['p'][j]; y = P['y'][j]
    fig.add_scatter(x=t, y=p[:,0,0], line=dict(width=0), showlegend=False, hoverinfo='skip', visible=vis, row=1, col=1)
    fig.add_scatter(x=t, y=p[:,0,2], fill='tonexty', fillcolor='rgba(44,163,220,.25)', line=dict(width=0), name='P10–P90', visible=vis, row=1, col=1)
    fig.add_scatter(x=t, y=p[:,0,1], line=dict(color='#2ca3dc', dash='dot'), name='pronóstico P50', visible=vis, row=1, col=1)
    fig.add_scatter(x=t, y=y[:,0], line=dict(color='white'), name='real', visible=vis, row=1, col=1)
    fig.add_scatter(x=t, y=y[:,4], line=dict(color='#ff8a5c'), name='entrada seco real', visible=vis, row=2, col=1)
    fig.add_scatter(x=t, y=p[:,4,1], line=dict(color='#ffd166', dash='dot'), name='entrada seco P50', visible=vis, row=2, col=1)
    fig.add_scatter(x=t, y=p[:,4,2], line=dict(color='#ffd166', width=1), name='entrada seco P90', visible=vis, row=2, col=1)
    fig.add_bar(x=t, y=dry[j].astype(int), marker_color=np.where(dry[j], '#7fd18b', '#2ca3dc'), name='decisión', visible=vis, row=3, col=1)
fig.add_hline(y=INLET_MAX, line_dash='dash', line_color='red', row=2, col=1)
per = 8
steps = []
for k, j in enumerate(days):
    v = [False]*(per*len(days)); v[k*per:(k+1)*per] = [True]*per
    steps.append(dict(method='update', args=[{'visible': v}], label=str(pd.to_datetime(P['times'][j][0]).date())))
fig.update_layout(sliders=[dict(active=0, steps=steps, currentvalue=dict(prefix='Día: '), pad=dict(t=30))], height=720,
                  template='plotly_dark', showlegend=False, title=f'{city} · 2026 (clima real) · 24 h por delante')
fig.show()

## 5 · ¿En qué se fija el modelo? (atención)
Cada fila es una hora futura a pronosticar (0 = dentro de 0 h … 23 = dentro de 23 h); cada columna, una hora pasada (−48 … −1). Brillante = el modelo mira más esa hora de historia.

In [7]:
P0 = prepare('Querétaro', tf, sc, dev)
hh = torch.tensor((P0['h'][:300] - sc.h[0]) / sc.h[1], device=dev); ff = torch.tensor((P0['f'][:300] - sc.f[0]) / sc.f[1], device=dev)
with torch.no_grad(): _, enc_w, cross_w = tf(hh, ff, return_attn=True)
A = cross_w[-1].mean((0, 1)).float().cpu().numpy()          # [24 futuro, 48 pasado]
fig = px.imshow(A, x=list(range(-LH, 0)), y=list(range(FH)), aspect='auto', color_continuous_scale='Viridis', template='plotly_dark',
                labels=dict(x='hora pasada (relativa al inicio del pronóstico)', y='hora futura pronosticada', color='atención'),
                title='Cross-attention (última capa, promedio de 4 cabezas y 300 ventanas)')
fig.update_layout(height=480); fig.show()
lag = A.mean(0); top = np.argsort(lag)[::-1][:6]
print('Horas pasadas con más atención:', [f'{t-LH}h' for t in top])

Horas pasadas con más atención: ['-8h', '-6h', '-7h', '-9h', '-31h', '-10h']


## 6 · Del pronóstico a la decisión: control híbrido hora por hora
**Regla:** usar modo seco en una hora si (a) el pronóstico de temperatura de entrada seca ≤ 27 °C y (b) el agua ahorrada vale más que la energía extra (λ = L de agua que "cuesta" 1 kWh extra; incluye el agua indirecta de la red eléctrica).
Cada política se prueba en **2026 con las 7 ciudades** contra el simulador (la verdad):
- **Siempre evaporativo** = status quo.
- **Reactiva** = regla que usa la hora anterior.
- **Ingenua** = repite lo de ayer (con ventaja: usa contrafactuales).
- **IA** = Transformer.
- **Oráculo** = conoce el futuro (techo teórico).

In [8]:
LAM = 1.0   # litros de agua equivalentes a 1 kWh extra (supuesto editable)
Ps = [prepare(c, tf, sc, dev) for c in CITIES]
res = pd.concat([evaluate(P, lam=LAM) for P in Ps])
ml = res[res.politica == POLICIES[3]].merge(clima[['ciudad','lat','lon','pct_seco_factible']], on='ciudad')
fig = px.scatter_map(ml, lat='lat', lon='lon', size='agua_ahorrada_l', color='agua_ahorrada_%', text='ciudad',
        color_continuous_scale='Tealgrn', size_max=55, zoom=4.3, center=dict(lat=24, lon=-102), map_style='carto-darkmatter',
        hover_data={'agua_ahorrada_%':':.0f','agua_ahorrada_l':':,.0f','energia_extra_%':':.1f','horas_riesgo_evitable':True,'lat':False,'lon':False},
        title='Agua ahorrada por la IA en 2026 (módulo de ~340 kW IT, 9 meses) · tamaño = litros, color = %')
fig.update_layout(height=580, margin=dict(l=0,r=0,t=50,b=0)); fig.show()

In [9]:
tot = res.groupby('politica')[['agua_l','energia_kwh','horas_riesgo_evitable']].sum().loc[POLICIES]
b = tot.iloc[0]
resumen = pd.DataFrame({'agua ahorrada %': 100*(b.agua_l-tot.agua_l)/b.agua_l, 'energía extra %': 100*(tot.energia_kwh-b.energia_kwh)/b.energia_kwh,
                        'horas inseguras evitables': tot.horas_riesgo_evitable})
display(resumen.round(1))
fig = px.bar(res, x='ciudad', y='agua_ahorrada_%', color='politica', barmode='group', template='plotly_dark',
             category_orders={'politica': POLICIES}, color_discrete_sequence=['#555','#8f9aa8','#ffb347','#2ca3dc','#7fd18b'],
             title='Agua ahorrada vs. siempre-evaporativo, por ciudad y política (2026)')
fig.update_layout(height=430); fig.show()

,agua ahorrada %,energía extra %,horas inseguras evitables
politica,,,
Siempre evaporativo (status quo),0.0,0.0,4134
Reactiva (regla con la hora anterior),56.4,5.6,1411
Pronóstico ingenuo (ayer),56.5,5.7,1833
IA: Transformer (pronóstico),56.3,5.5,839
Oráculo (techo teórico),56.1,5.5,0


### Ahorro vs. seguridad: la comparación justa
Una política puede "ahorrar más" simplemente siendo más arriesgada. Aquí barremos el margen de seguridad de cada política y miramos **agua ahorrada vs. horas inseguras evitables** (horas en que se eligió un modo que rebasa 27 °C habiendo otro que no). Más arriba y a la izquierda es mejor.

In [10]:
pf = pareto(Ps, margins=np.arange(-1.0, 4.01, 0.5), lam=LAM, q=1)
orc = pf[(pf.politica == POLICIES[4]) & (pf.margen_c == 0)]
lines = pf[~pf.politica.isin([POLICIES[0], POLICIES[4]])]
fig = px.line(lines, x='horas_riesgo_evitable', y='agua_ahorrada_%', color='politica', markers=True, hover_data=['margen_c'], template='plotly_dark',
              color_discrete_sequence=['#ffb347','#2ca3dc','#8f9aa8'],
              title='Frontera ahorro–seguridad (7 ciudades, 2026). Cada punto = un margen de seguridad distinto',
              labels={'horas_riesgo_evitable':'horas inseguras evitables (menos = mejor)', 'agua_ahorrada_%':'agua ahorrada (%)'})
fig.add_scatter(x=orc.horas_riesgo_evitable, y=orc['agua_ahorrada_%'], mode='markers+text', text=['oráculo'], textposition='middle right',
                marker=dict(symbol='star', size=16, color='#7fd18b'), name='Oráculo (techo, margen 0)')
fig.update_layout(height=470); fig.show()
a = pf[(pf.margen_c == 0)].set_index('politica')
print(a[['agua_ahorrada_%','horas_riesgo_evitable']].round(1).to_string())

                                       agua_ahorrada_%  horas_riesgo_evitable
politica                                                                     
IA: Transformer (pronóstico)                      56.3                    839
Oráculo (techo teórico)                           56.1                      0
Pronóstico ingenuo (ayer)                         56.5                   1833
Reactiva (regla con la hora anterior)             56.4                   1411
Siempre evaporativo (status quo)                   0.0                   4134


## 7 · ¿Y si hace más calor? ¿Y si el agua de la red pesa más?
**Izquierda:** con el mismo clima de 2026 más ΔT (escenarios de calentamiento; el modelo trabaja fuera de lo que vio, tómalo como estrés). **Derecha:** curva agua–energía al cambiar λ (cuánta agua vale 1 kWh extra).

In [11]:
rows = []
for dT in (0.0, 1.5, 3.0):
    for c in CITIES:
        Pd = prepare(c, tf, sc, dev, dT=dT)
        r = evaluate(Pd, lam=LAM); base, mlr = r.iloc[0], r[r.politica == POLICIES[3]].iloc[0]
        rows.append(dict(ciudad=c, dT=dT, agua_status_quo_l=base.agua_l, agua_ahorrada_pct=mlr['agua_ahorrada_%'], seco_h=mlr.horas_seco))
dtdf = pd.DataFrame(rows)
lam_rows = []
for lam in (0, .5, 1, 2, 3, 4, 6, 8):
    t = pd.concat([evaluate(P, lam=lam) for P in Ps]); g = t.groupby('politica')[['agua_l','energia_kwh']].sum()
    lam_rows.append(dict(lam=lam, agua_ahorrada=100*(1-g.loc[POLICIES[3],'agua_l']/g.loc[POLICIES[0],'agua_l']),
                         energia_extra=100*(g.loc[POLICIES[3],'energia_kwh']/g.loc[POLICIES[0],'energia_kwh']-1)))
lamdf = pd.DataFrame(lam_rows)
fig = make_subplots(rows=1, cols=2, subplot_titles=('Agua ahorrada por la IA según calentamiento', 'Curva agua–energía (λ)'))
for c in CITIES:
    d = dtdf[dtdf.ciudad == c]; fig.add_scatter(x=d.dT, y=d.agua_ahorrada_pct, mode='lines+markers', name=c, row=1, col=1)
fig.add_scatter(x=lamdf.energia_extra, y=lamdf.agua_ahorrada, mode='lines+markers+text', text=[f'λ={l}' for l in lamdf.lam],
                textposition='top center', showlegend=False, line_color='#7fd18b', row=1, col=2)
fig.update_xaxes(title_text='+ΔT (°C)', row=1, col=1); fig.update_yaxes(title_text='agua ahorrada (%)', row=1, col=1)
fig.update_xaxes(title_text='energía extra (%)', row=1, col=2); fig.update_yaxes(title_text='agua ahorrada (%)', row=1, col=2)
fig.update_layout(height=450, template='plotly_dark'); fig.show()
piv = dtdf.pivot(index='ciudad', columns='dT', values='agua_status_quo_l').round(0)
print('Agua del status quo (L) con más calor — crece donde el evaporativo ya trabaja duro:'); display(piv)

Agua del status quo (L) con más calor — crece donde el evaporativo ya trabaja duro:


dT,0.0,1.5,3.0
ciudad,,,
Chihuahua,"1,047,801.0","1,098,061.0","1,149,151.0"
Ciudad de México,"1,000,865.0","1,056,279.0","1,112,128.0"
Guadalajara,"1,152,784.0","1,209,367.0","1,266,685.0"
Mexicali,"1,186,068.0","1,235,011.0","1,284,328.0"
Monterrey,"1,272,867.0","1,331,972.0","1,391,648.0"
Mérida,"1,475,924.0","1,539,774.0","1,604,049.0"
Querétaro,"1,061,482.0","1,117,617.0","1,174,264.0"


## 8 · GreenAI: cuánto cuesta correr el modelo
Medido en este equipo (no una estimación de energía).

In [12]:
hh = torch.tensor((P0['h'][:1000]-sc.h[0])/sc.h[1], device=dev); ff = torch.tensor((P0['f'][:1000]-sc.f[0])/sc.f[1], device=dev)
for name, m in (('Transformer', tf), ('LSTM', ls)):
    with torch.no_grad():
        m(hh[:64], ff[:64]); torch.cuda.synchronize() if dev=='cuda' else None
        t0 = time.perf_counter()
        for _ in range(10): m(hh, ff)
        torch.cuda.synchronize() if dev=='cuda' else None
        dt = (time.perf_counter()-t0)/10
    print(f'{name}: {dt*1000:.1f} ms por 1000 ventanas ({dt*1e3/1000:.3f} ms/pronóstico de 24 h) en {dev}')

Transformer: 3.8 ms por 1000 ventanas (0.004 ms/pronóstico de 24 h) en cuda
LSTM: 2.3 ms por 1000 ventanas (0.002 ms/pronóstico de 24 h) en cuda


## 9 · Límites (léelos antes de presentar)
1. **Todo lo operativo es simulado.** La carga, los estados de la torre y los consumos salen de un simulador calibrado con un kit sintético. El modelo aprende **mi simulador**, no un centro de datos real. Lo que sí es real: el clima.
2. **Contrafactuales.** En la realidad no se observa qué habría pasado en el otro modo; aquí sí (es simulación). Con datos reales habría que estimar el modo no usado.
3. **Calibración con humedad supuesta.** El kit no trae humedad; se calibró suponiendo RH = 50 %.
4. **Un solo tipo de torre híbrida** y un límite térmico fijo (27 °C). En Mérida y Monterrey ni el modo evaporativo mantiene 27 °C muchas horas: ahí hace falta enfriamiento líquido, que este modelo no cubre.
5. **El pronóstico del clima usa ruido sintético** sobre el clima real, no un pronóstico meteorológico real.
6. **Agua = captación in situ.** El agua indirecta (generación eléctrica) entra solo vía λ, un supuesto editable.
7. **Ganancia modesta de la IA frente a una regla simple.** A igual ahorro de agua, la IA reduce las horas inseguras evitables (ver sección 6), pero queda lejos del oráculo. No es magia.
8. Los escenarios +ΔT son extrapolación para el modelo; sirven como prueba de estrés, no como predicción climática.

**Qué mediríamos antes de recomendar cambios reales:** humedad/bulbo húmedo horario en sitio, caudal y ciclos de concentración de la torre, y la intensidad hídrica de la electricidad local.